#  Script to demo the processing of the L Band of SAR

- It starts by setting up the directory paths and specifying parameters like the heading direction and polarimetric channel. 
- The script then loads various necessary datasets, including scene parameters, phase-to-height factors (kz), sensor trajectories, and digital terrain models (DTMs). 
- It reads Single Look Complex (SLC) images for the specified polarimetric channel and performs repeat-pass tomography to generate tomographic profiles both in the azimuth and ground range directions. 
- These profiles are visualized using matplotlib. Additionally, the script reads ancillary geocoding data from MetaSensing files, adjusting the data to match the scene parameters.
-  The processed data helps in understanding the vertical structure and spatial distribution of the observed scene in both azimuth and ground range dimensions.

1. **Scene Parameters**
   Scene parameters include metadata about the SAR acquisition, such as the number of samples (range dimension) and lines (azimuth dimension). These parameters are essential for correctly interpreting the SAR data and are usually stored in a separate file.

   - **NrTot**: Total number of range samples.
   - **NcTot**: Total number of azimuth lines.

2. **Phase-to-Height Factors (kz)**
   Phase-to-height factors, denoted as kz, are critical in SAR interferometry and tomography. They represent the relationship between the phase difference observed in SAR images and the height of the terrain. These factors are used to derive elevation information from phase data.

   - **kz**: A 3D array where each slice corresponds to the phase-to-height factors for a specific azimuth line and range sample.

3. **Sensor Trajectories**
   Sensor trajectories describe the flight path of the SAR platform (e.g., a satellite or an aircraft) during the data acquisition process. Accurate knowledge of the sensor's position and velocity is vital for precise geolocation and correction of motion-induced errors.

   - **Trajectories**: This includes information about the sensor's position (latitude, longitude, altitude) and i
\

Key Adjustments and Inline Comments (changes from matlab script) :
- Function Placeholder: The read_biomass_data function needs implementation based on the actual data format.
- Path Handling: Constructed using os.path.join for better path handling.
- Polarimetric and Heading Indices: Adjusted for 0-based indexing in Python.
- Scene Parameters: Loaded using scipy.io.loadmat.
- Tomographic Processing: Adjustments for handling complex numbers and numpy operations.
- Geocoding Data: Reading geocoding data from the NetCDF file using the netCDF4 library.
- Plotting: Visualization of tomographic data using matplotlib.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from netCDF4 import Dataset

# # Placeholder function to read biomass data (assumed to be implemented)
# def read_biomass_data(filepath, param, mode):
#     # Placeholder function
#     # You need to implement this based on the actual data format
#     pass

# Options

# Change directory to parent folder and set paths
os.chdir('..')
parent_path = os.path.join(os.getcwd(), 'SAR_data_Matlab', 'L')
os.chdir('Matlab_DEMO')

# Define heading and polarimetric channel
heading = 'NW'  # Either 'NW' (290°) or 'SE' (110°)
polarimetric_channel = 'HH'  # 'HH', 'HV' or 'VV'

# Determine the polarimetric index based on the channel
if polarimetric_channel == 'HH':
    pol_ind = 1
elif polarimetric_channel == 'HV':
    pol_ind = 2
elif polarimetric_channel == 'VV':
    pol_ind = 3

# Loading

# Loading scene parameters
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'SceneParameters.mat')
scene_parameters = loadmat(curr_fullpath)

ground_range_axis = scene_parameters['ground_range_axis'].squeeze()
azimuth_axis = scene_parameters['azimuth_axis'].squeeze()
Ngr = len(ground_range_axis)
Naz = len(azimuth_axis)

# Loading phase-to-height factors
kz = np.zeros((Ngr, Naz, scene_parameters['N'][0][0]), dtype=np.single)
for n in range(scene_parameters['N'][0][0]):
    curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'kz', scene_parameters['filenames_kz'][n][0])
    print(f'Loading {curr_fullpath}...')
    kz[:, :, n] = read_biomass_data(curr_fullpath, [], 2)
    print('done.')

# Loading sensor trajectories
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'FocusingCenters.mat')
focusing_centers = loadmat(curr_fullpath)

# Loading DTMs
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'MetaSensingDTM.mat')
meta_sensing_dtm = loadmat(curr_fullpath)
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'TomoDTMsr.mat')
tomo_dtm_sr = loadmat(curr_fullpath)

# Loading SLC images
I = np.zeros((Ngr, Naz, scene_parameters['N'][0][0]), dtype=np.single)
for n in range(scene_parameters['N'][0][0]):
    curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'PoliMi', scene_parameters['filenames_slc'][n, pol_ind-1][0])
    print(f'Loading {curr_fullpath}...')
    I[:, :, n] = read_biomass_data(curr_fullpath, [], 1)
    print('done.')

# Tomography

# Local vertical axis (with respect to the local ground level)
z_axis = np.arange(-60, 61, 1)
Nz = len(z_axis)

monostatic_tomography = True
if monostatic_tomography:
    n_tomo = scene_parameters['monostatic_acquisition'].astype(bool)
else:
    n_tomo = ~scene_parameters['monostatic_acquisition'].astype(bool)

# Azimuth profile
S = np.zeros((Nz, Naz))
gr_ind = round(Ngr / 4)
F = np.sum(np.abs(I[gr_ind, :, n_tomo])**2, axis=2)

for z_ind in range(Nz):
    z = z_axis[z_ind]

    # Ground steering
    Igs = I[gr_ind, :, n_tomo] * np.exp(-1j * kz[gr_ind, :, n_tomo] * (tomo_dtm_sr['TomoDTMsr'][gr_ind, :] - meta_sensing_dtm['MetaSensingDTM'][gr_ind, :]))

    S[z_ind, :] = np.abs(np.sum(Igs * np.exp(-1j * kz[gr_ind, :, n_tomo] * z), axis=2))**2 / (F + np.finfo(float).eps)

plt.figure()
plt.imshow(S, aspect='auto', extent=[azimuth_axis[0]*1e-3, azimuth_axis[-1]*1e-3, z_axis[-1], z_axis[0]], origin='lower')
plt.xlabel('Azimuth [km]')
plt.ylabel('[m]')
plt.title('Azimuth Profile')
plt.colorbar()
plt.show()

# Ground range profile
S = np.zeros((Nz, Ngr))
az_ind = round(Naz / 3)
F = np.sum(np.abs(I[:, az_ind, n_tomo])**2, axis=2)

for z_ind in range(Nz):
    z = z_axis[z_ind]

    # Ground steering
    Igs = I[:, az_ind, n_tomo] * np.exp(-1j * kz[:, az_ind, n_tomo] * (tomo_dtm_sr['TomoDTMsr'][:, az_ind] - meta_sensing_dtm['MetaSensingDTM'][:, az_ind]))

    S[z_ind, :] = np.abs(np.sum(Igs * np.exp(-1j * kz[:, az_ind, n_tomo] * z), axis=2))**2 / (F + np.finfo(float).eps)

plt.figure()
plt.imshow(S.T, aspect='auto', extent=[ground_range_axis[0]*1e-3, ground_range_axis[-1]*1e-3, z_axis[-1], z_axis[0]], origin='lower')
plt.xlabel('Ground range [km]')
plt.ylabel('[m]')
plt.title('Ground Range Profile')
plt.colorbar()
plt.show()

# Reading MetaSensing ancillary data
if heading == 'NW':
    curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200914160410_2_1.4G_HH_41_pres_16_fdc_0.sar.sig.nc')
elif heading == 'SE':
    curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200914150909_2_1.4G_HH_41_pres_16_fdc_0.sar.sig.nc')

with Dataset(curr_fullpath, 'r') as ds:
    LatImage_azimuth_axis = ds.variables['CrossRange'][:]
    LatImage_ground_range_axis = ds.variables['GroundRange'][:]
    LatImage = ds.variables['LatImage'][:]

# Determine indices for slicing the MetaSensing data to match the scene parameters
az_start_ind = np.argmin(np.abs(LatImage_azimuth_axis - azimuth_axis[0]))
az_stop_ind = np.argmin(np.abs(LatImage_azimuth_axis - azimuth_axis[-1]))
gr_start_ind = np.argmin(np.abs(LatImage_ground_range_axis - ground_range_axis[0]))
gr_stop_ind = np.argmin(np.abs(LatImage_ground_range_axis - ground_range_axis[-1]))

LatImage = LatImage[gr_start_ind:gr_stop_ind+1, az_start_ind:az_stop_ind+1]
